In [11]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import pandas as pd
import os
import logging
import numpy as np
import plotly.express as px
import warnings
import yaml
import matplotlib.pyplot as plt
import yaml

warnings.filterwarnings('ignore')
logging.getLogger('tensorboard').setLevel(logging.ERROR)
def fetch_events(log_dir, tag="val/acc"):
    event_acc = EventAccumulator(log_dir)
    event_acc.Reload()
    return event_acc

def find_experiments(path, list_of_runs):
    # loop through all runs to gather results
    runs = [i for i in np.sort(os.listdir(path)) if "." not in i]
    runs_all = list_of_runs.copy()
    for r_ in runs:
        r2 = [i for i in os.listdir(os.path.join(path, r_))]
        for r2_ in r2:
            runs_all.append(os.path.join(path, r_, r2_))
    return runs_all

list_of_runs = []
list_of_runs = find_experiments("../../logs", list_of_runs)

pandas_dataframe = pd.DataFrame()
plot_data = {}
idx = 0

exp_filter = [

            ############################# gmae ###################################
            ["vit_base_imagenet_test1/",                "gmae-base Gaussian-256"],  
            ["vit_base_imagenet/",                "gmae-base Gaussian-256"],  
            # ["vit_base_imagenet_test1_lp6/5",                "gmae-base Gaussian-256"],  
            ["vit_base_imagenet_test1_lp7/",                "gmae-base Gaussian-256"],  

            ["vit_base_imagenet_test1_ft3/",                "gmae-base Gaussian-256"],  
            
        
]   

for run in list_of_runs:
    for exp_ in exp_filter:
        if exp_[0] not in run:
            continue
        
        if not os.path.exists(run + "/results/"):
            continue

        try:
            log_dir = run + "/tensorboard/lightning_logs/version_0/"
            events = fetch_events(log_dir)
            try:
                a1 = events.Scalars("val/linear_probe_acc_11")
            except:
                a1 = events.Scalars("val/linear_probe_acc_11")

            values_with_steps = []
            c = 0
            for event in a1:
                c += 1
                values_with_steps.append((event.step, event.value))

            # [(event.step, event.value) for event in a1]
            values_with_steps = np.array(values_with_steps)
            # sort by steps
            values_with_steps = values_with_steps[values_with_steps[:,0].argsort()]
            # filter if steps are repeated
            values_with_steps = [values_with_steps[0]] + [values_with_steps[i] for i in range(1, len(values_with_steps)) if values_with_steps[i][0] != values_with_steps[i-1][0]]
            try:
                configs = yaml.load(open(run + "/.hydra/config.yaml", 'r'), Loader=yaml.FullLoader)
                mix_token_ratio =  configs['configs']['mix_token_ratio']
                mask_ratio =  configs['configs']['mask_ratio']
                mask_type =  configs['configs']['training_type']
            except Exception as e:
                print(e)
                mix_token_ratio =  configs['configs']['mix_token_ratio']
                mask_ratio =  configs['configs']['mask_ratio']
                mask_type =  "none"
            # if(mask_ratio>0):
            #     continue
            for e,(x,y) in enumerate(values_with_steps):
                ds = {"exp": exp_[1], "group": run, "exp_name": run, "step": e, "y": y, "mix_token_ratio": mix_token_ratio, "mask_ratio": mask_ratio, "mask_type": mask_type}
                pandas_dataframe = pandas_dataframe.append(ds, ignore_index=True)
        except Exception as e:
            # print(e)
            pass

fig = px.line(pandas_dataframe, x="step", y="y", color="exp_name", hover_data=["exp", "step", "y", "mix_token_ratio", "mask_ratio", "mask_type"], title="Imagenet", markers=True)
fig.update_layout(template="plotly_dark")
# fig.update_layout(yaxis_range=[0.0,0.85])
# fig.update_layout(xaxis_range=[0,400])
fig.update_layout(
    autosize=False,
    width=900,
    height=600, 
    margin=dict(t=24,l=0,b=16,r=30),
    showlegend=True,
)

# show labels at the end of each line
fig.update_traces(textposition='top center')

fig.show()